# 7 - Traçabilité des runs

Chaque opération d'écriture publique (`build_schema`, `update_database`, `add_columns`,
`delete_rows`, `delete_columns`) construit un `OperationReport` : un bilan mesuré via les
fonctions d'introspection propres de DuckLake (`ducklake_table_info`,
`ducklake_snapshots`, `ducklake_table_changes`), jamais estimé côté Python. Elle peut en
plus enregistrer, sur le snapshot DuckLake résultant, un `run_id` (l'auteur du commit),
un `commit_message` et un `commit_info` (JSON arbitraire fusionné dans
`commit_extra_info`) — de quoi retrouver, après coup, *quel* run de production a produit
*quel* état de la base.

Ce notebook ne ré-illustre pas les opérations elles-mêmes (`update_database` et
`is_categorical` : notebook 1 ; hiérarchies : notebook 5 ; `recluster`/`maintain` :
notebook 6) : il se concentre sur la traçabilité — écrire l'identité d'un run, lire son
rapport, puis relire l'historique des snapshots pour le retrouver.

### Table des matières

0. [Importation des modules](#section_0)
1. [Données synthétiques](#section_1)
2. [OperationReport et le commit DuckLake](#section_2)
3. [run_id / commit_message / commit_info à l'écriture](#section_3)
   - [build_schema](#section_3_1)
   - [update_database](#section_3_2)
   - [Les autres opérations](#section_3_3)
4. [Le rapport : summary(), to_dict(), warnings](#section_4)
5. [Relire l'historique : list_ducklake_snapshots](#section_5)
   - [Retrouver un run précis — par run_id, jamais par position](#section_5_1)
6. [Connexion sans catalogue DuckLake réel : dégradation gracieuse](#section_6)

## 0. Importation des modules <a id="section_0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys

import duckdb
import polars as pl

# Ajout du chemin vers le package
sys.path.append("..")

# Importation des modules ad hoc
from dt_ducklake_manager.connection import DuckLakeConnector
from dt_ducklake_manager.maintenance import DatabaseRecoveryManager
from dt_ducklake_manager.operations import DatabaseDeleter, DatabaseUpdater
from dt_ducklake_manager.schema import DuckLakeTablesBuilder

## 1. Données synthétiques <a id="section_1"></a>

In [ ]:
# Jeu de données minimal : un indicateur par modèle et par date
df = pl.DataFrame(
    {
        "model": ["model_A", "model_A", "model_B", "model_B"],
        "date": ["2026-01-01", "2026-01-02", "2026-01-01", "2026-01-02"],
        "value": [10.0, 11.0, 20.0, 21.0],
    }
)
PRIMARY_KEYS = ["model", "date"]

# Suppression du catalogue et des données existants pour garantir un état initial propre
CATALOG_PATH = os.path.join("../outputs", "run_traceability_demo.ducklake")
DATA_PATH = os.path.join("../outputs", "run_traceability_demo_data/")
for suffix in ["", ".wal"]:
    path_to_remove = CATALOG_PATH + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)

conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
df

## 2. OperationReport et le commit DuckLake <a id="section_2"></a>

`run_id`, `commit_message` et `commit_info` sont enregistrés **dans la même transaction**
que l'opération elle-même, juste avant le `COMMIT`, via `ducklake_set_commit_message` : le
snapshot DuckLake qui en résulte porte alors ces informations comme `author`,
`commit_message` et `commit_extra_info` — visibles ensuite dans `ducklake_snapshots(...)`.
Ce trio est accepté, à l'identique, par les cinq opérations d'écriture publiques.

## 3. run_id / commit_message / commit_info à l'écriture <a id="section_3"></a>

### 3.1 build_schema <a id="section_3_1"></a>

`build_schema` retourne directement un `OperationReport`.

In [ ]:
builder = DuckLakeTablesBuilder(
    df,
    primary_keys=PRIMARY_KEYS,
    connection=conn,
)
report_build = builder.build_schema(
    run_id="build-2026-09-18",
    commit_message="Construction initiale du jeu de données",
    commit_info={"pipeline": "demo-notebook-7"},
)
print(report_build.summary())
print(f"run_id enregistré sur le rapport : {report_build.run_id}")

### 3.2 update_database <a id="section_3_2"></a>

`update_database` retourne un booléen : le rapport détaillé est exposé via `updater.last_report`, pas via la valeur de retour.

In [ ]:
updater = DatabaseUpdater(connection=conn)

# Mise à jour de deux lignes existantes
update_31 = df.filter(pl.col("model") == "model_A").with_columns(
    value=pl.Series([15.0, 16.0])
)

success_31 = updater.update_database(
    update_df=update_31,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_transaction=True,
    run_id="update-2026-09-18-A",
    commit_message="Rafraîchissement quotidien - model_A",
    commit_info={"pipeline": "demo-notebook-7", "rows": len(update_31)},
)

report_update_a = updater.last_report
print(f"Succès : {success_31}")
print(report_update_a.summary())

### 3.3 Les autres opérations <a id="section_3_3"></a>

`add_columns`, `delete_rows` et `delete_columns` acceptent exactement le même trio
`run_id`/`commit_message`/`commit_info` et retournent directement leur
`OperationReport` — voir les notebooks 1 (`add_columns`/`delete_rows`) et 5
(`delete_columns`) pour leur usage détaillé ; le mécanisme de traçabilité lui-même
est identique à celui montré ci-dessus.

## 4. Le rapport : summary(), to_dict(), warnings <a id="section_4"></a>

`OperationReport.summary()` produit une ligne lisible ; `to_dict()` le rend
JSON-sérialisable (utile pour l'expédier vers un système de logs/monitoring). Un
échec n'empêche pas le rapport d'exister : il reste attaché à `self.last_report`,
partiel, avec le message d'erreur ajouté à `warnings`.

In [ ]:
import json

print(report_update_a.summary())
print()
print(json.dumps(report_update_a.to_dict(), indent=2, default=str, ensure_ascii=False))

In [ ]:
# report.warnings : une colonne absente lors d'une suppression n'est jamais passée
# sous silence, elle est nommée dans warnings plutôt que de lever une exception
deleter = DatabaseDeleter(connection=conn)
report_bad_delete = deleter.delete_columns(
    ["colonne_inexistante"], use_transaction=False
)

print(f"columns_dropped : {report_bad_delete.columns_dropped}")
print(f"warnings        : {report_bad_delete.warnings}")

## 5. Relire l'historique : list_ducklake_snapshots <a id="section_5"></a>

`DatabaseRecoveryManager.list_ducklake_snapshots()` liste l'intégralité de
l'historique des snapshots du catalogue (toutes les colonnes exposées par
`ducklake_snapshots`, y compris `author`, `commit_message` et `commit_extra_info`),
trié par `snapshot_id` décroissant.

In [ ]:
# Un second run, pour peupler un peu plus l'historique
update_31b = df.filter(pl.col("model") == "model_B").with_columns(
    value=pl.Series([25.0, 26.0])
)
updater.update_database(
    update_df=update_31b,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_transaction=True,
    run_id="update-2026-09-18-B",
    commit_message="Rafraîchissement quotidien - model_B",
)

recovery = DatabaseRecoveryManager(connection=conn)
snapshots = recovery.list_ducklake_snapshots()
# .to_native() donne un pyarrow.Table (backend de list_ducklake_snapshots) : conversion
# vers polars pour la sélection/le filtrage, comme au notebook 1 (§4.5)
snapshots_pl = snapshots.to_polars()
snapshots_pl[["snapshot_id", "author", "commit_message", "commit_extra_info"]]

### 5.1 Retrouver un run précis — par run_id, jamais par position <a id="section_5_1"></a>

`update_database` lance par défaut une compaction après commit
(`compact_after_update=True`) : elle s'exécute comme un commit **séparé**, y compris
quand elle ne change rien, et ce commit n'a ni `author` ni `commit_message`. Le
snapshot d'un run donné n'est donc **pas forcément le dernier de l'historique** : on
le retrouve en filtrant sur `author` (le `run_id` fourni), jamais sur la position de
la ligne.

In [ ]:
# Le dernier snapshot de l'historique n'est pas forcément le nôtre : il peut être un
# commit de compaction, sans auteur
print("Dernier snapshot :")
print(snapshots_pl[["snapshot_id", "author", "commit_message"]].head(1))

# On retrouve le run 'update-2026-09-18-A' par son author, pas par sa position
our_run = snapshots_pl.filter(pl.col("author") == "update-2026-09-18-A")
print("\nRun 'update-2026-09-18-A' retrouvé par author :")
print(our_run[["snapshot_id", "author", "commit_message", "commit_extra_info"]])

## 6. Connexion sans catalogue DuckLake réel : dégradation gracieuse <a id="section_6"></a>

Sur une connexion sans catalogue DuckLake réellement attaché (typiquement une
connexion DuckDB in-memory, comme dans les tests unitaires), les mesures
DuckLake-only (`snapshot_before`/`after`, `files_*`, `bytes_*`) restent à leur valeur
par défaut et `ducklake_set_commit_message` est silencieusement ignoré (log DEBUG) —
sans lever d'exception. Le `run_id` fourni par l'appelant, lui, reste présent sur le
rapport : c'est un champ Python simple, jamais mesuré.

In [ ]:
conn_memory = duckdb.connect(":memory:")
builder_memory = DuckLakeTablesBuilder(
    df, primary_keys=PRIMARY_KEYS, connection=conn_memory
)
report_memory = builder_memory.build_schema(run_id="build-sans-catalogue")

print(f"run_id sur le rapport   : {report_memory.run_id}")
print(
    f"snapshot_before/after   : {report_memory.snapshot_before} / "
    f"{report_memory.snapshot_after}"
)

conn_memory.close()
conn.close()